# Capstone 2 — Automated Code Review Agent

**Book:** [Agentic AI: Build, Ship, Portfolio](../index.html)

---

In this capstone you will build an **automated PR review agent** that:

1. Parses code changes using AST analysis.
2. Runs static analysis for code quality issues.
3. Scans for security vulnerabilities.
4. Checks adherence to style guidelines.
5. Generates natural language feedback.
6. Orchestrates a full PR review pipeline.
7. Integrates with GitHub via a mock interface.

> **Note:** This notebook uses *mock/simulated* LLM responses so it runs without an API key. Replace the mock functions with real LLM calls for production use.

---
## 1 Setup

In [ ]:
# --- Setup ---

import ast
import json
import re
import textwrap
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any

print("All imports ready.")

In [ ]:
# --- Mock LLM Helper ---

def mock_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """Simulate an LLM response based on keywords in the prompt."""
    lower = (system_prompt + user_prompt).lower()
    if "security" in lower and ("scan" in lower or "vulnerab" in lower):
        return json.dumps({
            "findings": [
                {"severity": "HIGH", "type": "SQL Injection",
                 "line": 15, "description": "User input concatenated directly into SQL query.",
                 "suggestion": "Use parameterized queries instead of string concatenation."},
                {"severity": "MEDIUM", "type": "Hardcoded Secret",
                 "line": 3, "description": "API key appears to be hardcoded.",
                 "suggestion": "Move secrets to environment variables or a secret manager."}
            ]
        })
    elif "style" in lower or "convention" in lower:
        return json.dumps({
            "issues": [
                {"line": 5, "rule": "E501", "message": "Line too long (95 > 88 characters)"},
                {"line": 12, "rule": "W291", "message": "Trailing whitespace"},
                {"line": 8, "rule": "N802", "message": "Function name should be lowercase"}
            ]
        })
    elif "feedback" in lower or "review comment" in lower:
        return ("## Code Review Feedback\n\n"
                "### Summary\n"
                "This PR introduces a new user authentication module. Overall the logic is sound, "
                "but there are security concerns that must be addressed before merging.\n\n"
                "### Critical Issues\n"
                "1. **SQL Injection (line 15):** User input is concatenated into a SQL query. "
                "Use parameterized queries.\n"
                "2. **Hardcoded API key (line 3):** Move to environment variables.\n\n"
                "### Suggestions\n"
                "- Add input validation for email format.\n"
                "- Consider adding rate limiting to the login endpoint.\n\n"
                "### Verdict: REQUEST_CHANGES")
    elif "static" in lower or "quality" in lower:
        return json.dumps({
            "issues": [
                {"type": "unused_variable", "line": 7, "name": "temp_data",
                 "message": "Variable 'temp_data' is assigned but never used."},
                {"type": "complexity", "line": 20, "name": "process_request",
                 "message": "Function has cyclomatic complexity of 12 (threshold: 10)."},
                {"type": "missing_return_type", "line": 20, "name": "process_request",
                 "message": "Function missing return type annotation."}
            ]
        })
    else:
        return "Mock response: Analysis complete."

print("Mock LLM helper ready.")

In [ ]:
# --- Sample code to review ---

SAMPLE_PR_CODE = textwrap.dedent("""\
import os
import sqlite3
API_KEY = "sk-secret-key-12345"

def GetUserByEmail(email):
    conn = sqlite3.connect("users.db")
    temp_data = []
    cursor = conn.cursor()
    # Build query
    query = "SELECT * FROM users WHERE email = '" + email + "'"
    cursor.execute(query)
    result = cursor.fetchone()
    conn.close()
    return result

def process_request(request_data):
    if request_data.get("type") == "login":
        email = request_data["email"]
        password = request_data["password"]
        user = GetUserByEmail(email)
        if user:
            if user[2] == password:
                if request_data.get("remember"):
                    if request_data.get("device"):
                        return {"status": "ok", "token": "abc123"}
                    else:
                        return {"status": "ok", "token": "abc123"}
                else:
                    return {"status": "ok", "token": "temp"}
            else:
                return {"status": "error", "message": "Wrong password"}
        else:
            return {"status": "error", "message": "User not found"}
    elif request_data.get("type") == "register":
        return {"status": "ok", "message": "Registered"}
    else:
        return {"status": "error", "message": "Unknown request type"}
""")

print(f"Sample PR code: {len(SAMPLE_PR_CODE)} characters, {len(SAMPLE_PR_CODE.splitlines())} lines")
print(SAMPLE_PR_CODE)

---
## 2 Code Parser (AST Analysis)

The **Code Parser** uses Python's built-in `ast` module to analyze code structure. It extracts functions, classes, imports, and complexity metrics without executing the code.

In [ ]:
@dataclass
class FunctionInfo:
    """Metadata about a parsed function."""
    name: str
    line_number: int
    args: list[str]
    has_return_type: bool
    has_docstring: bool
    line_count: int
    complexity: int  # Simplified cyclomatic complexity


@dataclass
class ParseResult:
    """Result of parsing a code file."""
    imports: list[str]
    functions: list[FunctionInfo]
    classes: list[str]
    global_variables: list[str]
    total_lines: int
    syntax_valid: bool
    syntax_error: str | None = None


class CodeParser:
    """Parses Python code using AST analysis."""

    def _count_complexity(self, node: ast.FunctionDef) -> int:
        """Count branching statements as a proxy for cyclomatic complexity."""
        complexity = 1  # Base complexity
        for child in ast.walk(node):
            if isinstance(child, (ast.If, ast.While, ast.For, ast.ExceptHandler,
                                  ast.With, ast.Assert)):
                complexity += 1
            elif isinstance(child, ast.BoolOp):
                complexity += len(child.values) - 1
        return complexity

    def parse(self, code: str) -> ParseResult:
        """Parse Python source code and extract structural information."""
        lines = code.splitlines()
        try:
            tree = ast.parse(code)
        except SyntaxError as e:
            return ParseResult(
                imports=[], functions=[], classes=[], global_variables=[],
                total_lines=len(lines), syntax_valid=False,
                syntax_error=str(e)
            )

        imports = []
        functions = []
        classes = []
        global_vars = []

        for node in ast.iter_child_nodes(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    imports.append(alias.name)
            elif isinstance(node, ast.ImportFrom):
                imports.append(f"{node.module}")
            elif isinstance(node, ast.FunctionDef):
                has_docstring = (isinstance(node.body[0], ast.Expr) and
                                 isinstance(node.body[0].value, ast.Constant) and
                                 isinstance(node.body[0].value.value, str))
                end_line = node.end_lineno or node.lineno
                fi = FunctionInfo(
                    name=node.name,
                    line_number=node.lineno,
                    args=[a.arg for a in node.args.args],
                    has_return_type=node.returns is not None,
                    has_docstring=has_docstring,
                    line_count=end_line - node.lineno + 1,
                    complexity=self._count_complexity(node)
                )
                functions.append(fi)
            elif isinstance(node, ast.ClassDef):
                classes.append(node.name)
            elif isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        global_vars.append(target.id)

        return ParseResult(
            imports=imports,
            functions=functions,
            classes=classes,
            global_variables=global_vars,
            total_lines=len(lines),
            syntax_valid=True
        )

print("CodeParser defined.")

In [ ]:
# --- Test the Code Parser ---

parser = CodeParser()
parse_result = parser.parse(SAMPLE_PR_CODE)

print(f"Syntax valid: {parse_result.syntax_valid}")
print(f"Total lines:  {parse_result.total_lines}")
print(f"Imports:      {parse_result.imports}")
print(f"Global vars:  {parse_result.global_variables}")
print(f"Classes:      {parse_result.classes}")
print(f"Functions:")
for f in parse_result.functions:
    print(f"  - {f.name}(line {f.line_number}): args={f.args}, "
          f"complexity={f.complexity}, docstring={f.has_docstring}, "
          f"return_type={f.has_return_type}")

---
## 3 Static Analysis Agent

The **Static Analysis Agent** checks code quality: unused variables, high complexity, missing type annotations, overly long functions, and other code smells.

In [ ]:
class Severity(Enum):
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"
    CRITICAL = "CRITICAL"


@dataclass
class ReviewFinding:
    """A single finding from any analysis agent."""
    agent: str
    severity: Severity
    finding_type: str
    line: int | None
    message: str
    suggestion: str


class StaticAnalysisAgent:
    """Analyzes code for quality issues using AST + LLM."""

    def __init__(self, max_complexity: int = 10, max_function_lines: int = 50):
        self.max_complexity = max_complexity
        self.max_function_lines = max_function_lines

    def analyze(self, code: str, parse_result: ParseResult) -> list[ReviewFinding]:
        """Run static analysis on the code."""
        findings: list[ReviewFinding] = []

        # Check 1: Syntax errors
        if not parse_result.syntax_valid:
            findings.append(ReviewFinding(
                agent="StaticAnalysis", severity=Severity.CRITICAL,
                finding_type="syntax_error", line=None,
                message=f"Syntax error: {parse_result.syntax_error}",
                suggestion="Fix the syntax error before further review."
            ))
            return findings

        # Check 2: Function complexity
        for func in parse_result.functions:
            if func.complexity > self.max_complexity:
                findings.append(ReviewFinding(
                    agent="StaticAnalysis", severity=Severity.WARNING,
                    finding_type="high_complexity", line=func.line_number,
                    message=f"Function '{func.name}' has complexity {func.complexity} "
                            f"(threshold: {self.max_complexity}).",
                    suggestion="Break the function into smaller sub-functions."
                ))

        # Check 3: Missing docstrings
        for func in parse_result.functions:
            if not func.has_docstring:
                findings.append(ReviewFinding(
                    agent="StaticAnalysis", severity=Severity.INFO,
                    finding_type="missing_docstring", line=func.line_number,
                    message=f"Function '{func.name}' has no docstring.",
                    suggestion="Add a docstring describing the function's purpose and parameters."
                ))

        # Check 4: Missing return type annotations
        for func in parse_result.functions:
            if not func.has_return_type:
                findings.append(ReviewFinding(
                    agent="StaticAnalysis", severity=Severity.INFO,
                    finding_type="missing_return_type", line=func.line_number,
                    message=f"Function '{func.name}' has no return type annotation.",
                    suggestion="Add a return type hint (e.g., -> dict | None)."
                ))

        # Check 5: LLM-assisted deeper analysis
        llm_raw = mock_llm(
            system_prompt="You are a static code quality analyzer.",
            user_prompt=f"Analyze this code for quality issues:\n```python\n{code}\n```"
        )
        llm_data = json.loads(llm_raw)
        for issue in llm_data.get("issues", []):
            findings.append(ReviewFinding(
                agent="StaticAnalysis",
                severity=Severity.WARNING,
                finding_type=issue["type"],
                line=issue.get("line"),
                message=issue["message"],
                suggestion=f"Review {issue.get('name', 'this code')} for improvements."
            ))

        print(f"[StaticAnalysis] Found {len(findings)} issues")
        return findings

print("StaticAnalysisAgent defined.")

In [ ]:
# --- Test Static Analysis ---

static_agent = StaticAnalysisAgent(max_complexity=10)
static_findings = static_agent.analyze(SAMPLE_PR_CODE, parse_result)

for f in static_findings:
    print(f"  [{f.severity.value}] Line {f.line}: {f.message}")

---
## 4 Security Scanner Agent

The **Security Scanner** looks for common vulnerability patterns: SQL injection, hardcoded secrets, unsafe deserialization, command injection, and more.

In [ ]:
class SecurityScannerAgent:
    """Scans code for security vulnerabilities using pattern matching + LLM."""

    # Regex patterns for common vulnerabilities
    PATTERNS = {
        "hardcoded_secret": {
            "pattern": r'(?:API_KEY|SECRET|PASSWORD|TOKEN)\s*=\s*["\'][^"\']+ ["\']',
            "severity": Severity.ERROR,
            "message": "Potential hardcoded secret detected.",
            "suggestion": "Use environment variables or a secrets manager."
        },
        "sql_injection": {
            "pattern": r'["\']\s*\+\s*\w+\s*\+\s*["\']',
            "severity": Severity.CRITICAL,
            "message": "Possible SQL injection via string concatenation.",
            "suggestion": "Use parameterized queries (cursor.execute('SELECT ... WHERE x = ?', (val,)))."
        },
        "eval_usage": {
            "pattern": r'\beval\s*\(',
            "severity": Severity.CRITICAL,
            "message": "Use of eval() detected.",
            "suggestion": "Avoid eval(). Use ast.literal_eval() for safe parsing or refactor."
        },
        "pickle_usage": {
            "pattern": r'pickle\.loads?\(',
            "severity": Severity.ERROR,
            "message": "Unsafe deserialization with pickle detected.",
            "suggestion": "Use JSON or a safe serialization format for untrusted data."
        }
    }

    def scan(self, code: str) -> list[ReviewFinding]:
        """Scan code for security vulnerabilities."""
        findings: list[ReviewFinding] = []
        lines = code.splitlines()

        # Rule-based scanning
        for vuln_type, config in self.PATTERNS.items():
            for i, line in enumerate(lines, 1):
                if re.search(config["pattern"], line):
                    findings.append(ReviewFinding(
                        agent="SecurityScanner",
                        severity=config["severity"],
                        finding_type=vuln_type,
                        line=i,
                        message=config["message"],
                        suggestion=config["suggestion"]
                    ))

        # LLM-assisted deeper scan
        llm_raw = mock_llm(
            system_prompt="You are a security vulnerability scanner.",
            user_prompt=f"Scan this code for security vulnerabilities:\n```python\n{code}\n```"
        )
        llm_data = json.loads(llm_raw)
        for finding in llm_data.get("findings", []):
            # Avoid duplicates with rule-based findings
            existing_lines = {f.line for f in findings}
            if finding.get("line") not in existing_lines:
                findings.append(ReviewFinding(
                    agent="SecurityScanner",
                    severity=Severity.ERROR if finding["severity"] == "HIGH" else Severity.WARNING,
                    finding_type=finding["type"].lower().replace(" ", "_"),
                    line=finding.get("line"),
                    message=finding["description"],
                    suggestion=finding["suggestion"]
                ))

        print(f"[SecurityScanner] Found {len(findings)} vulnerabilities")
        return findings

print("SecurityScannerAgent defined.")

In [ ]:
# --- Test Security Scanner ---

security_agent = SecurityScannerAgent()
security_findings = security_agent.scan(SAMPLE_PR_CODE)

for f in security_findings:
    print(f"  [{f.severity.value}] Line {f.line}: {f.finding_type} - {f.message}")

---
## 5 Style Checker Agent

The **Style Checker** enforces naming conventions, line length, formatting rules, and project-specific guidelines.

In [ ]:
class StyleCheckerAgent:
    """Checks code style and conventions."""

    def __init__(self, max_line_length: int = 88):
        self.max_line_length = max_line_length

    def check(self, code: str, parse_result: ParseResult) -> list[ReviewFinding]:
        """Check code against style rules."""
        findings: list[ReviewFinding] = []
        lines = code.splitlines()

        # Check 1: Line length
        for i, line in enumerate(lines, 1):
            if len(line) > self.max_line_length:
                findings.append(ReviewFinding(
                    agent="StyleChecker", severity=Severity.INFO,
                    finding_type="line_too_long", line=i,
                    message=f"Line is {len(line)} chars (max: {self.max_line_length}).",
                    suggestion="Break the line or refactor for readability."
                ))

        # Check 2: Trailing whitespace
        for i, line in enumerate(lines, 1):
            if line != line.rstrip():
                findings.append(ReviewFinding(
                    agent="StyleChecker", severity=Severity.INFO,
                    finding_type="trailing_whitespace", line=i,
                    message="Trailing whitespace detected.",
                    suggestion="Remove trailing whitespace."
                ))

        # Check 3: Function naming (PEP 8: snake_case)
        for func in parse_result.functions:
            if func.name != func.name.lower():
                findings.append(ReviewFinding(
                    agent="StyleChecker", severity=Severity.WARNING,
                    finding_type="naming_convention", line=func.line_number,
                    message=f"Function '{func.name}' should be snake_case per PEP 8.",
                    suggestion=f"Rename to '{self._to_snake_case(func.name)}'."
                ))

        # Check 4: Global variable naming (PEP 8: UPPER_CASE for constants)
        for var in parse_result.global_variables:
            if var != var.upper() and not var.startswith("_"):
                findings.append(ReviewFinding(
                    agent="StyleChecker", severity=Severity.INFO,
                    finding_type="constant_naming", line=None,
                    message=f"Global variable '{var}' may be a constant — consider UPPER_CASE.",
                    suggestion=f"If '{var}' is a constant, rename to '{var.upper()}'."
                ))

        print(f"[StyleChecker] Found {len(findings)} style issues")
        return findings

    @staticmethod
    def _to_snake_case(name: str) -> str:
        """Convert CamelCase to snake_case."""
        result = re.sub(r'([A-Z])', r'_\1', name).lower().lstrip('_')
        return result

print("StyleCheckerAgent defined.")

In [ ]:
# --- Test Style Checker ---

style_agent = StyleCheckerAgent(max_line_length=88)
style_findings = style_agent.check(SAMPLE_PR_CODE, parse_result)

for f in style_findings:
    print(f"  [{f.severity.value}] Line {f.line}: {f.message}")

---
## 6 Natural Language Feedback Generator

The **Feedback Generator** takes all findings from the analysis agents and produces a human-readable review comment, organized by severity and with actionable suggestions.

In [ ]:
class ReviewVerdict(Enum):
    APPROVE = "APPROVE"
    REQUEST_CHANGES = "REQUEST_CHANGES"
    COMMENT = "COMMENT"


@dataclass
class ReviewResult:
    """Complete review result."""
    verdict: ReviewVerdict
    summary: str
    findings: list[ReviewFinding]
    feedback_markdown: str
    stats: dict[str, int]


class FeedbackGenerator:
    """Generates natural language review feedback from findings."""

    def _determine_verdict(self, findings: list[ReviewFinding]) -> ReviewVerdict:
        """Determine the review verdict based on severity of findings."""
        severities = [f.severity for f in findings]
        if Severity.CRITICAL in severities or severities.count(Severity.ERROR) >= 2:
            return ReviewVerdict.REQUEST_CHANGES
        elif Severity.ERROR in severities:
            return ReviewVerdict.REQUEST_CHANGES
        elif Severity.WARNING in severities:
            return ReviewVerdict.COMMENT
        else:
            return ReviewVerdict.APPROVE

    def _compute_stats(self, findings: list[ReviewFinding]) -> dict[str, int]:
        """Compute summary statistics."""
        stats = {"total": len(findings)}
        for sev in Severity:
            stats[sev.value] = sum(1 for f in findings if f.severity == sev)
        return stats

    def generate(self, findings: list[ReviewFinding], code: str) -> ReviewResult:
        """Generate a complete review with natural language feedback."""
        verdict = self._determine_verdict(findings)
        stats = self._compute_stats(findings)

        # Group by severity
        grouped: dict[str, list[ReviewFinding]] = {}
        for sev in [Severity.CRITICAL, Severity.ERROR, Severity.WARNING, Severity.INFO]:
            group = [f for f in findings if f.severity == sev]
            if group:
                grouped[sev.value] = group

        # Build markdown feedback
        md_parts = [f"## Code Review — {verdict.value}\n"]
        md_parts.append(f"**{stats['total']} issues found** | "
                        f"Critical: {stats.get('CRITICAL', 0)} | "
                        f"Error: {stats.get('ERROR', 0)} | "
                        f"Warning: {stats.get('WARNING', 0)} | "
                        f"Info: {stats.get('INFO', 0)}\n")

        for sev_name, group in grouped.items():
            md_parts.append(f"### {sev_name}\n")
            for f in group:
                line_ref = f"(line {f.line})" if f.line else ""
                md_parts.append(f"- **{f.finding_type}** {line_ref}: {f.message}")
                md_parts.append(f"  - *Suggestion:* {f.suggestion}\n")

        feedback_md = "\n".join(md_parts)

        # Use LLM for a natural summary
        summary = mock_llm(
            system_prompt="You are a code reviewer. Write a brief summary as a review comment.",
            user_prompt=f"Generate feedback for a PR with {stats['total']} issues."
        )

        result = ReviewResult(
            verdict=verdict,
            summary=summary[:300],
            findings=findings,
            feedback_markdown=feedback_md,
            stats=stats
        )

        print(f"[FeedbackGenerator] Verdict: {verdict.value}, "
              f"Issues: {stats['total']}")
        return result

print("FeedbackGenerator defined.")

In [ ]:
# --- Test Feedback Generator ---

all_findings = static_findings + security_findings + style_findings
feedback_gen = FeedbackGenerator()
review_result = feedback_gen.generate(all_findings, SAMPLE_PR_CODE)

print(review_result.feedback_markdown)

---
## 7 PR Review Pipeline

The **ReviewPipeline** orchestrates all agents in sequence: parse, static analysis, security scan, style check, then feedback generation.

In [ ]:
class ReviewPipeline:
    """Orchestrates the full code review pipeline."""

    def __init__(self):
        self.parser = CodeParser()
        self.static_agent = StaticAnalysisAgent(max_complexity=10)
        self.security_agent = SecurityScannerAgent()
        self.style_agent = StyleCheckerAgent(max_line_length=88)
        self.feedback_gen = FeedbackGenerator()

    def review(self, code: str, filename: str = "unknown.py",
               verbose: bool = True) -> ReviewResult:
        """Run the full review pipeline on a code string."""
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Reviewing: {filename}")
            print(f"{'='*60}")

        # Step 1: Parse
        if verbose:
            print("\n--- Step 1: Parse ---")
        parse_result = self.parser.parse(code)
        if verbose:
            print(f"  Syntax valid: {parse_result.syntax_valid}, "
                  f"Functions: {len(parse_result.functions)}, "
                  f"Lines: {parse_result.total_lines}")

        # Step 2: Static Analysis
        if verbose:
            print("\n--- Step 2: Static Analysis ---")
        static = self.static_agent.analyze(code, parse_result)

        # Step 3: Security Scan
        if verbose:
            print("\n--- Step 3: Security Scan ---")
        security = self.security_agent.scan(code)

        # Step 4: Style Check
        if verbose:
            print("\n--- Step 4: Style Check ---")
        style = self.style_agent.check(code, parse_result)

        # Step 5: Generate Feedback
        if verbose:
            print("\n--- Step 5: Generate Feedback ---")
        all_findings = static + security + style
        result = self.feedback_gen.generate(all_findings, code)

        if verbose:
            print(f"\n{'='*60}")
            print(f"  Review Complete: {result.verdict.value}")
            print(f"{'='*60}")

        return result

print("ReviewPipeline defined.")

In [ ]:
# --- Run the full pipeline ---

pipeline = ReviewPipeline()
result = pipeline.review(SAMPLE_PR_CODE, filename="auth.py")

print("\n" + "=" * 60)
print("FULL REVIEW FEEDBACK:")
print("=" * 60)
print(result.feedback_markdown)

---
## 8 GitHub Integration (Mock)

In production, the review agent would integrate with GitHub's API to:
- Fetch PR diffs.
- Post review comments.
- Submit reviews with a verdict.

Below we mock this integration.

In [ ]:
@dataclass
class PullRequest:
    """Mock Pull Request."""
    pr_number: int
    title: str
    author: str
    branch: str
    files: dict[str, str]  # filename -> code content
    status: str = "open"


@dataclass
class PRComment:
    """A review comment on a PR."""
    pr_number: int
    body: str
    verdict: str
    posted_at: str = field(default_factory=lambda: datetime.now().isoformat())


class MockGitHubClient:
    """Simulates GitHub API interactions."""

    def __init__(self):
        self.comments: list[PRComment] = []

    def get_pr_files(self, pr: PullRequest) -> dict[str, str]:
        """Get changed files from a PR."""
        print(f"[GitHub] Fetching files for PR #{pr.pr_number}: {pr.title}")
        return pr.files

    def post_review(self, pr: PullRequest, review: ReviewResult) -> PRComment:
        """Post a review comment on a PR."""
        comment = PRComment(
            pr_number=pr.pr_number,
            body=review.feedback_markdown,
            verdict=review.verdict.value
        )
        self.comments.append(comment)
        print(f"[GitHub] Posted review on PR #{pr.pr_number}: {review.verdict.value}")
        return comment


def review_pull_request(pr: PullRequest) -> dict[str, ReviewResult]:
    """Review all files in a PR."""
    github = MockGitHubClient()
    pipeline = ReviewPipeline()
    files = github.get_pr_files(pr)

    results = {}
    for filename, code in files.items():
        if filename.endswith(".py"):
            result = pipeline.review(code, filename=filename)
            results[filename] = result
            github.post_review(pr, result)

    return results

print("MockGitHubClient defined.")

In [ ]:
# --- Simulate a PR review ---

mock_pr = PullRequest(
    pr_number=42,
    title="Add user authentication module",
    author="developer123",
    branch="feature/auth",
    files={"auth.py": SAMPLE_PR_CODE}
)

pr_results = review_pull_request(mock_pr)

for filename, result in pr_results.items():
    print(f"\n{filename}: {result.verdict.value} ({result.stats['total']} issues)")

---
## 9 Domain Variants

The code review agent architecture adapts to different codebases and languages by swapping out parsers, rules, and prompts.

In [ ]:
@dataclass
class ReviewDomainConfig:
    """Domain-specific review configuration."""
    name: str
    language: str
    max_line_length: int
    max_complexity: int
    custom_rules: list[str]
    security_focus: list[str]

REVIEW_DOMAINS = {
    "web_api": ReviewDomainConfig(
        name="Web API Review",
        language="python",
        max_line_length=88,
        max_complexity=10,
        custom_rules=["All endpoints must have authentication",
                      "Rate limiting required on public endpoints",
                      "Input validation on all user-facing parameters"],
        security_focus=["SQL injection", "XSS", "CSRF", "auth bypass"]
    ),
    "data_pipeline": ReviewDomainConfig(
        name="Data Pipeline Review",
        language="python",
        max_line_length=120,
        max_complexity=15,
        custom_rules=["All transforms must be idempotent",
                      "Schema validation at pipeline boundaries",
                      "Logging at each transformation step"],
        security_focus=["PII exposure", "data leakage", "insecure storage"]
    ),
    "ml_model": ReviewDomainConfig(
        name="ML Model Code Review",
        language="python",
        max_line_length=100,
        max_complexity=12,
        custom_rules=["Random seeds must be set for reproducibility",
                      "Model artifacts must have versioning",
                      "Training/inference paths must be clearly separated"],
        security_focus=["model poisoning", "data exfiltration", "unsafe pickle"]
    )
}

for name, cfg in REVIEW_DOMAINS.items():
    print(f"\n{cfg.name}")
    print(f"  Language: {cfg.language}, Max line: {cfg.max_line_length}, Max complexity: {cfg.max_complexity}")
    print(f"  Rules: {cfg.custom_rules[:2]}...")
    print(f"  Security: {cfg.security_focus}")

---
## 10 Exercises

### Exercise 1 (Conceptual)

The current pipeline runs all analysis agents sequentially. In a real-world scenario with large PRs (50+ files), this can be slow.

**Question:** Design an architecture where:
- Files are analyzed in parallel.
- Each file is routed to the appropriate language-specific parser.
- A prioritization agent decides which files to review first (e.g., security-sensitive files first).
- Results are aggregated into a single coherent review.

Describe the agent topology, communication protocol, and how you would handle partial failures.

---

### Exercise 2 (Coding)

Implement an `AutoFixer` agent that takes a `ReviewFinding` and attempts to generate a corrected version of the code.

```python
@dataclass
class CodeFix:
    original_line: int
    original_code: str
    fixed_code: str
    explanation: str
    confidence: float  # 0.0 - 1.0

class AutoFixerAgent:
    def fix(self, code: str, finding: ReviewFinding) -> CodeFix | None:
        # Your implementation here
        pass

    def fix_all(self, code: str, findings: list[ReviewFinding]) -> list[CodeFix]:
        # Apply fixes in reverse line order to avoid offset issues
        pass
```

Test it on the sample PR code. Only apply fixes with confidence > 0.8.

---

### Exercise 3 (Design)

Design a **learning review agent** that improves over time:

- When a developer dismisses a review comment, the agent records this as a false positive.
- When a developer accepts and fixes based on a comment, it is a true positive.
- Over time the agent adjusts its severity thresholds and rule weights.
- The agent can also learn team-specific conventions from accepted PRs.

Write a design document covering:
1. Feedback data model (what to store per interaction)
2. How to update rule weights without full retraining
3. How to detect and adapt to team-specific conventions
4. Privacy and fairness considerations

In [ ]:
# --- Scratch cell for exercises ---

print("Ready for exercises.")